In [ ]:
import pandas as pd
#DATA CLEANING
#POSITIVE DATASET
#read csv file
df = pd.read_csv("epitope_table_export_1772462355.csv.zip")

#read the column having peptide seq that is Epitope.1
df = df[['Epitope.1']]

#renamed the column topeptide
df = df.rename(columns={'Epitope.1': 'peptide'})

#changed the rowsequence by pushing it one line up
df = df.iloc[1:]
df = df.reset_index(drop=True)

#created label column showing 1 that is positive
df['label'] = 1
df.head()
len(df)

#removed duplicates
#print("Before removing duplicates:", len(df))

df = df.drop_duplicates(subset=['peptide'])

#print("After removing duplicates:", len(df))

#removed invalid sequence
import re

#print("Before cleaning:", len(df))

df = df[df['peptide'].str.match("^[ACDEFGHIKLMNPQRSTVWY]+$")]

#print("After removing invalid sequences:", len(df))

df['length'] = df['peptide'].apply(len)
#df['length'].describe()

#only kept the peptides with length between 8 to 25
df = df[(df['length'] >= 8) & (df['length'] <= 25)]
df = df.drop(columns=['length'])

#print("Final positive dataset size:", len(df))

#downloaded cleaned data csv file
#df.to_csv("positive_cleaned_epitopes.csv", index=False)
#from google.colab import files
#files.download("positive_cleaned_epitopes.csv")



#NEGATIVE DATASET
#installed biopython
!pip install biopython

#file is in gz format soimported gzip
import gzip

#imported SeqIO to read fasta file
from Bio import SeqIO

#created empty list to read all proteinseq
sequences = []

#opened fasta file and with helpof rt read thefile as text as file namedhandle
with gzip.open("uniprotkb_organism_id_9606_AND_reviewed_2026_03_02.fasta.gz", "rt") as handle:

  #read fasta file using SeqIO recordgivesprotein seq only
    for record in SeqIO.parse(handle, "fasta"):

      #record.seq gave amino acidsseq
        sequences.append(str(record.seq))

#print("Total proteins:", len(sequences))

#creating the 15-mer peptide only
negative_peptides = set()

for seq in sequences:
    if len(seq) >= 15:
        for i in range(len(seq) - 15 + 1):
            peptide = seq[i:i+15]
            negative_peptides.add(peptide)

#print("Total unique 15-mers generated:", len(negative_peptides))

#remove overlapping or same peptides from both positive and negative dataset
# Convert positive peptides to set for fast lookup
positive_set = set(df['peptide'])

#print("Total positives:", len(positive_set))

# Remove positives from negative set
negative_filtered = negative_peptides - positive_set

#print("Negatives after removing overlap:", len(negative_filtered))

#now balance the negative as same number as positivee by removing random entities
import random

negative_sample = random.sample(list(negative_filtered), 5597)

#print("Sampled negatives:", len(negative_sample))

#now convert the sampled negatives which was in last format to table or dataframe format

df_neg = pd.DataFrame({
    "peptide": negative_sample,
    "label": 0
})

df_neg.head()

#combine both dataset positive and negative
df_final = pd.concat([df, df_neg])

#shuffle dataset
df_final = df_final.sample(frac=1).reset_index(drop=True)
#check
#print(df_final.shape)
df_final.head()
df_final = df_final[df_final['peptide'].apply(len) == 15]

# Add an additional cleaning step for df_final to remove non-standard amino acids
#print(f"Shape of df_final before amino acid filtering: {df_final.shape}")
initial_df_final_count = len(df_final)
df_final = df_final[df_final['peptide'].str.match("^[ACDEFGHIKLMNPQRSTVWY]+$")]
removed_count = initial_df_final_count - len(df_final)
if removed_count > 0:
    print(f"Removed {removed_count} peptides containing non-standard amino acids.")
#print(f"Shape of df_final after amino acid filtering: {df_final.shape}")

#MACHINE LEARNING
#feature encoding converting peptides letters in numbers for ML

import numpy as np

amino_acids = "ACDEFGHIKLMNPQRSTVWY"
aa_dict = {aa: i for i, aa in enumerate(amino_acids)}

def encode_peptide(seq):
    encoding = []
    for aa in seq:
        vector = [0]*20
        vector[aa_dict[aa]] = 1
        encoding.extend(vector)
    return encoding

X = np.array([encode_peptide(seq) for seq in df_final['peptide']])
y = df_final['label'].values

#print(X.shape)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#training model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

from sklearn.metrics import classification_report
#print(classification_report(y_test, y_pred))
#checking accuracy of model

from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

#print("Accuracy:", accuracy_score(y_test, y_pred))
#print(classification_report(y_test, y_pred))

#improving efficiency

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

from sklearn.metrics import classification_report
#print(classification_report(y_test, y_pred_rf))

#decided final model
LogisticRegression(class_weight='balanced')

#saving model
import joblib
joblib.dump(model, "autoimmune_model.pkl")

#confusion matrix
from sklearn.metrics import confusion_matrix
#print(confusion_matrix(y_test, y_pred))


#finally using model
import numpy as np

# take input from user
peptide = input("Enter a peptide sequence (length 15): ").upper()

# check length
if len(peptide) != 15:
    print("Error: Peptide must be exactly 15 amino acids long")
else:
    try:
        encoded = np.array(encode_peptide(peptide)).reshape(1, -1)

        prediction = model.predict(encoded)

        if prediction[0] == 1:
            print("AUTOIMMUNE")
        else:
            print("NON-AUTOIMMUNE")

    except:
        print("Error: Invalid amino acids used")





Enter a peptide sequence (length 15): QWERTYIPASDFGHK
NON-AUTOIMMUNE
